# Pandas 🐼

<pre>Algumas organizações oferecem bibliotecas para facilitar a criação de consultas, integrando-as ao nosso código.</pre>

<pre>O Banco Mundial oferece biblioteca em Python para consultar seus bancos de dados.</pre>

👉 mais detalhes, consulte o pacote <a href='https://pypi.org/project/wbgapi/'>wbgapi</a> e sua <a href='https://blogs.worldbank.org/opendata/introducing-wbgapi-new-python-package-accessing-world-bank-data'>documentação</a>.

<pre>Vamos instalar e importar as bibliotecas que serão utilizadas neste caderno.</pre>

In [1]:
!pip install wbgapi --quiet
!pip install pandas --quiet

In [2]:
import wbgapi as wb
import pandas as pd

<pre>Podemos listar as diversas fontes de dados disponíveis com a função abaixo.</pre>

In [3]:
wb.source.info()

id,name,code,concepts,lastupdated
1,Doing Business,DBS,3,2021-08-18
2,World Development Indicators,WDI,3,2025-04-15
3,Worldwide Governance Indicators,WGI,3,2024-11-05
5,Subnational Malnutrition Database,SNM,3,2016-03-21
6,International Debt Statistics,IDS,4,2025-02-26
11,Africa Development Indicators,ADI,3,2013-02-22
12,Education Statistics,EDS,3,2024-06-25
13,Enterprise Surveys,ESY,3,2022-03-25
14,Gender Statistics,GDS,3,2025-04-17
15,Global Economic Monitor,GEM,3,2025-04-16


<pre>Vamos consultar as séries de dados do banco <b>Gender Statistics</b> abaixo.</pre>

In [4]:
wb.series.info(db=14)

id,value
account.t.d,Account (% age 15+)
account.t.d.1,"Account, female (% age 15+)"
account.t.d.2,"Account, male (% age 15+)"
borrow.any,Borrowed any money (% age 15+)
borrow.any.1,"Borrowed any money, female (% age 15+)"
borrow.any.2,"Borrowed any money, male (% age 15+)"
fin1.1a,First financial institution account ever was opened to receive a wage payment (% age 15+)
fin1.1ab,First financial institution account ever was opened to receive a wage payment or money from the government (% age 15+)
fin1.1b,First financial institution ever account was opened to receive money from the government (% age 15+)
fin1.t.d,Financial institution account (% age 15+)


<pre>As tabelas <b>SL.TLF.ACTI.FE.ZS</b> e <b>SL.TLF.ACTI.MA.ZS</b> se referem à parcela da população feminina e masculina, respectivamente, de 15 a 64 anos que integra a força de trabalho.</pre>

<pre>A função de consulta <b>wb.data.DataFrame()</b> retorna, por padrão, um <b>DataFrame</b> em que os índices são o código ISO3 do país, da região ou do grupo de renda, bem como apresenta os anos correspondentes como colunas.</pre>

<pre>Vamos consultar as séries indicadas restringindo o período a um ano apenas, e transformá-las em <b>Series</b>.</pre>

In [5]:
female_labor_force = wb.data.DataFrame('SL.TLF.ACTI.FE.ZS', time = 2019).squeeze()
male_labor_force = wb.data.DataFrame('SL.TLF.ACTI.MA.ZS', time = 2019).squeeze()

In [6]:
female_labor_force

economy
ABW          NaN
AFE    62.875108
AFG    19.062000
AFW    66.349798
AGO    74.079000
         ...    
XKX          NaN
YEM     5.269000
ZAF    56.550000
ZMB    52.614000
ZWE    62.086000
Name: SL.TLF.ACTI.FE.ZS, Length: 266, dtype: float64

In [7]:
male_labor_force

economy
ABW          NaN
AFE    75.350881
AFG    70.659000
AFW    78.087791
AGO    77.909000
         ...    
XKX          NaN
YEM    61.357000
ZAF    68.534000
ZMB    67.291000
ZWE    72.977000
Name: SL.TLF.ACTI.MA.ZS, Length: 266, dtype: float64

<pre>Descubra a taxa de participação na força de trabalho feminina e masculina para o Brasil (código <b>BRA</b>), bem como para o mundo (código <b>WLD</b>), conforme solicitado pela enumeração a seguir.</pre>

👉 dica: utilize as Series `female_labor_force` e `male_labor_force` acima.

1. Participação feminina no Brasil.

<details>
  <summary>Resposta</summary>

<br/>

```python
f"{female_labor_force['BRA']:.2f}%"
```

<br/>

</details>

2. Participação masculina no Brasil.

<details>
  <summary>Resposta</summary>

<br/>

```python
f"{male_labor_force['BRA']:.2f}%"
```

<br/>

</details>

3. Participação feminina no mundo.

<details>
  <summary>Resposta</summary>

<br/>

```python
f"{female_labor_force['WLD']:.2f}%"
```

<br/>

</details>

4. Participação masculina no mundo.

<details>
  <summary>Resposta</summary>

<br/>

```python
f"{male_labor_force['WLD']:.2f}%"
```

<br/>

</details>

<pre>Como vimos, as linhas e índices da série incluem países, bem como agregações por região e por grupo de renda.</pre>

<pre>Antes de prosseguir com a análise, vamos separá-los para vocês.</pre>

In [8]:
economy_info = wb.economy.info()
region_info = wb.region.info()
income_info = wb.income.info()

In [9]:
country_codes = [pais.get('id') 
                 for pais in economy_info.items
                 if pais.get('region') != ''] # códigos que não são países têm o campo region e income em branco

In [10]:
income_levels = [item.get('id') for item in income_info.items]
income_levels

['HIC', 'INX', 'LIC', 'LMC', 'LMY', 'MIC', 'UMC']

In [11]:
paises_info = [{'id': pais.get('id'),
                'name': pais.get('value'),
                'region': pais.get('region'),
                'income_level': pais.get('incomeLevel')}
                for pais in economy_info.items 
                if pais.get("region") != ""]

<pre>Com os objetos que já temos, crie as séries <b>female_labor_force_income</b> e <b>male_labor_force_income</b>, ambas obtidas por filtragem por <b>grupos de renda</b> das séries completas (<b>female_labor_force</b> e <b>male_labor_force</b>, respectivamente).</pre>

👉 dica: para esse filtro, use os códigos dos grupos de renda (armazenados na variável <b>income_levels</b>) como índice das `Series`.

<details>
  <summary>Resposta</summary>

<br/>

```python
female_labor_force_income = female_labor_force[income_levels]
male_labor_force_income = male_labor_force[income_levels]
```

<br/>

</details>

<pre>Em seguida, crie as séries <b>female_labor_force_country</b> e <b>male_labor_force_country</b>, somente com códigos de países (armazenados na variável <b>country_codes</b>), ambas obtidas por filtragem por <b>países/regiões</b> das séries completas (<b>female_labor_force</b> e <b>male_labor_force</b>, respectivamente).

👉 dica: para esse filtro, use os códigos dos países/regiões (armazenados na variável <b>country_codes</b>) como índice das `Series`.

<details>
  <summary>Resposta</summary>

<br/>

```python
female_labor_force_country = female_labor_force[country_codes]
male_labor_force_country = male_labor_force[country_codes]
```

<br/>

</details>

<pre>Agora, junte as séries em dois <b>DataFrames</b>:</pre>

 - um <b>DataFrame</b> chamado <b>labor_force_income</b>, com as participações feminina (a ser armazenada na coluna <b>female_rate</b>) e masculina (a ser armazenada na coluna <b>male_rate</b>) por grupo de renda, e

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_income = pd.DataFrame({"female_rate": female_labor_force_income,
                                   "male_rate": male_labor_force_income})
```

<br/>

</details>

 - outro <b>DataFrame</b> chamado <b>labor_force_country</b>, com as participações feminina (a ser armazenada na coluna <b>female_rate</b>) e masculina (a ser armazenada na coluna <b>male_rate</b>) por região/país.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country = pd.DataFrame({"female_rate": female_labor_force_country,
                                    "male_rate": male_labor_force_country})
```

<br/>

</details>

<pre>Adicione também, em cada DataFrame criado, uma coluna chamada <b>rate_gap</b> com a diferença entre as taxas <b>male_rate</b> e <b>female_rate</b> presentes no DataFrame correspondente.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_income['rate_gap'] = labor_force_income["male_rate"] - labor_force_income["female_rate"]
labor_force_country['rate_gap'] = labor_force_country["male_rate"] - labor_force_country["female_rate"]
```

<br/>

</details>

<pre>Encontre a média, mediana, mínimo e máximo das participações masculina (<b>male_rate</b>) e feminina (<b>female_rate</b>), bem como da diferença (<b>rate_gap</b>), dentre os países.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.mean()
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.median(axis = "index")
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.min(axis = "index")
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.max(axis = "index")
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_income['rate_gap'].min()
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_income[labor_force_income['rate_gap'] == labor_force_income['rate_gap'].max()]
```

<br/>

</details>

<pre>Encontre o grupo de renda com menor diferença entre as taxas de participação na força de trabalho.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_income['rate_gap'].idxmin()
```

<br/>

</details>

<pre>Encontre a média da taxa de participação feminina na força de trabalho entre os países Brasil, Argentina, Uruguai e Paraguai.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
paises = [pais.get('id') 
          for pais in paises_info 
          if pais.get('name') in ['Brazil', 'Argentina', 'Uruguay', 'Paraguay']]
          
female_labor_force_country[paises].mean()
```

<br/>

</details>

<pre>Por fim, encontre a média da taxa de participação feminina na força de trabalho entre os países Canadá e Estados Unidos.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
[pais.get('id') 
          for pais in paises_info 
          if pais.get('name') in ['United States', 'Canada']]
          
female_labor_force_country[paises].mean()
```

<br/>

</details>